<a href="https://colab.research.google.com/github/Rayan1156/phishing_email_detection/blob/main/Phishing_Email_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import sklearn

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

!pip install -q datasets

from datasets import load_dataset

ds = load_dataset("mriadur/phishing-email-dataset")
print(ds)


pandas: 2.2.3
scikit-learn: 1.6.1
DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})


In [4]:
df = ds["train"].to_pandas()

print("Shape:", df.shape)
print(df["Email Type"].value_counts())
print(df.head(3))


Shape: (18650, 3)
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64
   Unnamed: 0                                         Email Text  Email Type
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...  Safe Email
1           1  the other side of * galicismos * * galicismo *...  Safe Email
2           2  re : equistar deal tickets are you still avail...  Safe Email


In [5]:
print("Missing texts:", df["Email Text"].isna().sum())
print("Junk rows:", df["Email Text"].str.strip().isin(["null", "empty", ""]).sum())
print(df["Email Text"].iloc[0][:400])


Missing texts: 16
Junk rows: 536
re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative are very thought-provoking , but i am not sure that it is fair to attribute this to " sons " being " treated like senior relatives " . for one thing , we do n't normally use ' brother ' in this way any more than we do 'd aughter ' , and it is hard to


In [6]:
df = df.drop(columns=["Unnamed: 0"])

df = df.dropna(subset=["Email Text"])
df = df[~df["Email Text"].str.strip().str.lower().isin(["null", "empty", ""])]
df = df.drop_duplicates(subset=["Email Text"])

df["label"] = df["Email Type"].map({"Safe Email": 0, "Phishing Email": 1})

print("Clean shape:", df.shape)
print(df["label"].value_counts())
print("Missing labels:", df["label"].isna().sum())


Clean shape: (17535, 3)
label
0    10979
1     6556
Name: count, dtype: int64
Missing labels: 0


In [7]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    df["Email Text"], df["label"],
    test_size=0.10, random_state=42, stratify=df["label"]
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1111, random_state=42, stratify=y_temp
)

print("Train:", len(X_train), " Val:", len(X_val), " Test:", len(X_test))
print("Phishing % — train:", round(y_train.mean(), 3),
      " val:", round(y_val.mean(), 3),
      " test:", round(y_test.mean(), 3))


Train: 14027  Val: 1754  Test: 1754
Phishing % — train: 0.374  val: 0.374  test: 0.374


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000))
])

model_lr.fit(X_train, y_train)
pred = model_lr.predict(X_val)

print("Accuracy :", round(accuracy_score(y_val, pred), 4))
print("Precision:", round(precision_score(y_val, pred), 4))
print("Recall   :", round(recall_score(y_val, pred), 4))
print("F1       :", round(f1_score(y_val, pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_val, pred))


Accuracy : 0.9778
Precision: 0.9753
Recall   : 0.9649
F1       : 0.9701
Confusion matrix:
 [[1082   16]
 [  23  633]]


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import pandas as pd

def evaluate(name, clf):
    m = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("clf", clf)
    ])
    m.fit(X_train, y_train)
    p = m.predict(X_val)
    models[name] = m
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_val, p), 4),
        "Precision": round(precision_score(y_val, p), 4),
        "Recall": round(recall_score(y_val, p), 4),
        "F1": round(f1_score(y_val, p), 4),
    }

models = {}
results = []
results.append(evaluate("Logistic Regression", LogisticRegression(max_iter=1000)))
results.append(evaluate("Naive Bayes", MultinomialNB()))
results.append(evaluate("Random Forest", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)))
results.append(evaluate("Linear SVM", LinearSVC()))

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print(results_df)


                 Model  Accuracy  Precision  Recall      F1
3           Linear SVM    0.9800     0.9755  0.9710  0.9733
0  Logistic Regression    0.9778     0.9753  0.9649  0.9701
2        Random Forest    0.9743     0.9796  0.9512  0.9652
1          Naive Bayes    0.9658     0.9715  0.9360  0.9534


In [10]:
from sklearn.pipeline import FeatureUnion

def evaluate2(name, clf):
    combined = FeatureUnion([
        ("word", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("char", TfidfVectorizer(analyzer="char_wb", max_features=5000, ngram_range=(3, 5))),
    ])
    m = Pipeline([("features", combined), ("clf", clf)])
    m.fit(X_train, y_train)
    p = m.predict(X_val)
    models[name + " (word+char)"] = m
    return {
        "Model": name + " (word+char)",
        "Accuracy": round(accuracy_score(y_val, p), 4),
        "Precision": round(precision_score(y_val, p), 4),
        "Recall": round(recall_score(y_val, p), 4),
        "F1": round(f1_score(y_val, p), 4),
    }

results2 = []
results2.append(evaluate2("Logistic Regression", LogisticRegression(max_iter=1000)))
results2.append(evaluate2("Naive Bayes", MultinomialNB()))
results2.append(evaluate2("Random Forest", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)))
results2.append(evaluate2("Linear SVM", LinearSVC()))

results2_df = pd.DataFrame(results2).sort_values("F1", ascending=False)
print(results2_df)


                             Model  Accuracy  Precision  Recall      F1
3           Linear SVM (word+char)    0.9852     0.9802  0.9802  0.9802
0  Logistic Regression (word+char)    0.9806     0.9814  0.9665  0.9739
2        Random Forest (word+char)    0.9766     0.9695  0.9680  0.9687
1          Naive Bayes (word+char)    0.9669     0.9476  0.9649  0.9562


In [11]:
import re
import numpy as np
from sklearn.preprocessing import FunctionTransformer, StandardScaler

URGENCY = ["urgent", "immediately", "verify", "suspended", "expires",
           "winner", "prize", "claim", "congratulations", "act now",
           "final notice", "limited"]

def indicator_features(texts):
    rows = []
    for t in texts:
        t = str(t)
        rows.append([
            t.count("http") + t.count("www"),                    # link count
            len(re.findall(r"\d\s\.\s\d\s\.\s\d\s\.\s\d", t)),   # IP-address pattern
            sum(t.count(w) for w in URGENCY),                    # urgency hits
            t.count("!"),                                        # exclamation count
            t.count("$") + t.count("£"),                         # money symbols
            sum(c.isdigit() for c in t) / max(len(t), 1),        # digit ratio
            len(t.split()),                                      # word count
        ])
    return np.array(rows, dtype=float)


In [12]:
def evaluate3(name, features, clf):
    m = Pipeline([("features", features), ("clf", clf)])
    m.fit(X_train, y_train)
    p = m.predict(X_val)
    models[name] = m
    return {"Model": name,
            "Accuracy": round(accuracy_score(y_val, p), 4),
            "Precision": round(precision_score(y_val, p), 4),
            "Recall": round(recall_score(y_val, p), 4),
            "F1": round(f1_score(y_val, p), 4)}

indicators_only = Pipeline([
    ("fn", FunctionTransformer(indicator_features)),
    ("scale", StandardScaler())
])

def hybrid_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("char", TfidfVectorizer(analyzer="char_wb", max_features=5000, ngram_range=(3, 5))),
        ("indicators", Pipeline([
            ("fn", FunctionTransformer(indicator_features)),
            ("scale", StandardScaler())
        ])),
    ])

results3 = []
results3.append(evaluate3("SVM (indicators only)", indicators_only, LinearSVC()))
results3.append(evaluate3("LR (indicators only)", indicators_only, LogisticRegression(max_iter=1000)))
results3.append(evaluate3("SVM (hybrid)", hybrid_features(), LinearSVC()))
results3.append(evaluate3("LR (hybrid)", hybrid_features(), LogisticRegression(max_iter=1000)))

print(pd.DataFrame(results3).sort_values("F1", ascending=False))


                   Model  Accuracy  Precision  Recall      F1
2           SVM (hybrid)    0.9846     0.9787  0.9802  0.9794
3            LR (hybrid)    0.9800     0.9770  0.9695  0.9732
1   LR (indicators only)    0.7457     0.8144  0.4146  0.5495
0  SVM (indicators only)    0.7195     0.8417  0.3079  0.4509


In [13]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

def wc_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("char", TfidfVectorizer(analyzer="char_wb", max_features=5000, ngram_range=(3, 5))),
    ])

def wc_model(clf):
    return Pipeline([("features", wc_features()), ("clf", clf)])

ensemble = VotingClassifier(
    estimators=[
        ("svm", wc_model(CalibratedClassifierCV(LinearSVC()))),
        ("lr",  wc_model(LogisticRegression(max_iter=1000))),
        ("rf",  wc_model(RandomForestClassifier(n_estimators=200, n_jobs=-1))),
        ("nb",  wc_model(MultinomialNB())),
    ],
    voting="soft",
)

ensemble.fit(X_train, y_train)
pred = ensemble.predict(X_val)
proba = ensemble.predict_proba(X_val)[:, 1]

print("Accuracy :", round(accuracy_score(y_val, pred), 4))
print("Precision:", round(precision_score(y_val, pred), 4))
print("Recall   :", round(recall_score(y_val, pred), 4))
print("F1       :", round(f1_score(y_val, pred), 4))
print(confusion_matrix(y_val, pred))
models["Ensemble (word+char)"] = ensemble


Accuracy : 0.9869
Precision: 0.9803
Recall   : 0.9848
F1       : 0.9825
[[1085   13]
 [  10  646]]


In [14]:
print("Thresh  Precision  Recall   F1      Missed scams")
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    p = (proba >= t).astype(int)
    fn = ((y_val == 1) & (p == 0)).sum()
    print(f"{t:.2f}    {precision_score(y_val, p):.4f}    {recall_score(y_val, p):.4f}  {f1_score(y_val, p):.4f}   {fn}")


Thresh  Precision  Recall   F1      Missed scams
0.30    0.9461    0.9909  0.9680   6
0.35    0.9545    0.9909  0.9723   6
0.40    0.9629    0.9893  0.9759   7
0.45    0.9774    0.9878  0.9826   8
0.50    0.9803    0.9848  0.9825   10
0.55    0.9861    0.9741  0.9801   17
0.60    0.9890    0.9634  0.9761   24
